# 12 · Buffalo evidence map: a rule-based combination

> **Not recommended, kept as a record of a test.** In the held-out camera test (section 2), combining the sources ranked hidden cameras **worse** than the model alone (rho 0.44 vs 0.56). The "average over available sources" rule also favours places with *less* data. Use the map in `10_buffalo_bias_checks.ipynb` instead.

No single source is good enough on its own (06–11), so this map **combines three kinds of evidence with fixed rules**, trusting each as much as it has earned:

| score | source | trusted because | limitation |
|---|---|---|---|
| **camera** | Snapshot Serengeti camera traps | effort and real absences | central ~1,100 km² only, 2010–2013 |
| **sightings** | GBIF sightings, effort-adjusted | covers every visited area | tourist bias, few records per place |
| **model** | XGBoost on embeddings + covariates (10) | fills gaps from landscape patterns | learned partly roads and the visited core |

**Rule:** each score becomes a percentile (0–100) over the Serengeti grid, so the three are comparable. The combined score is their **weighted average over the scores available at that place**: camera 0.5, model 0.3, sightings 0.2. Where there's no camera nearby, the camera weight drops out and the other two share the total. A **confidence** count says how many sources stand behind each place.

The cameras are the most trusted source, which also makes them the only data we could test against. So the rule is tested with **camera regions held out**: scores are built from the cameras in three regions and checked on the fourth.

Only the visited Serengeti (protected areas, visited cells) is mapped, where the model was tested. Everything comes from the cache built by 08–10.

In [1]:
import ee
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
from sklearn.cluster import KMeans

import buffalo_data as bd
import buffalo_validation as bv
from buffalo_data import BAND_COLS, COVARIATE_COLS, BuffaloConfig, build_buffalo_dataset

EE_PROJECT = "project-0cf410a3-f35f-4911-9c5"
ee.Initialize(project=EE_PROJECT)

config = BuffaloConfig(cache_dir="cache")
data = build_buffalo_dataset(config)
areas = bv.fetch_protected_areas()

MAP_YEAR = 2025
GRID_SPACING_M = 2_500

points = bv.add_utm(data.sdm_df, config.cell_size_m)
points["area"] = bv.protected_area_of(points, areas)
serengeti, _ = bv.year_matched(points[points["area"].isin(bv.SERENGETI_AREAS)])
serengeti = bd.cached(config, "serengeti_covariates", lambda: bd.add_covariates(serengeti, "point_id", config))
cam = bv.add_utm(data.cameras, config.cell_size_m)
cam = bd.cached(config, "camera_covariates", lambda: bd.add_covariates(cam, "SiteID", config))
grid = bd.cached(config, f"serengeti_grid_{MAP_YEAR}_{GRID_SPACING_M}",
                 lambda: bv.serengeti_map_grid(data.visited_cells, areas, config, MAP_YEAR, GRID_SPACING_M))
grid = grid.dropna(subset=BAND_COLS + COVARIATE_COLS).reset_index(drop=True)

records = bd.fetch_mammal_records(config)
records["x"], records["y"] = bd.to_utm.transform(records["decimalLongitude"].values, records["decimalLatitude"].values)
records["buffalo"] = (records["speciesKey"] == 2441034).astype(float)
print(f"grid: {len(grid)} points; cameras: {len(cam)}; GBIF mammal records: {len(records)} ({int(records['buffalo'].sum())} buffalo)")

presences: loaded from cache\buffalo_presences_8be5ffc111.parquet
buffalo sightings in box: 674
presences after thinning: 658
mammals: loaded from cache\buffalo_mammals_8be5ffc111.parquet
mammal sightings in box: 15873
visited cells: 334


points_embedded: loaded from cache\buffalo_points_embedded_8be5ffc111.parquet
points missing embeddings: 0
cameras_embedded: loaded from cache\buffalo_cameras_embedded_8be5ffc111.parquet
dataset: 658 presences, 6580 background, 222 cameras


serengeti_covariates: loaded from cache\buffalo_serengeti_covariates_8be5ffc111.parquet
camera_covariates: loaded from cache\buffalo_camera_covariates_8be5ffc111.parquet
serengeti_grid_2025_2500: loaded from cache\buffalo_serengeti_grid_2025_2500_8be5ffc111.parquet
mammal_records: loaded from cache\buffalo_mammal_records_8be5ffc111.parquet
mammal records in box: 15873
grid: 2693 points; cameras: 222; GBIF mammal records: 15873 (705 buffalo)


## 1. The three scores

Every score is computed in meters (ADR-0001) and turned into a **percentile over the Serengeti grid**: 90 means "higher than 90 % of the grid on this score".

- **model:** 10's final model (Serengeti only, regularized XGBoost, embeddings + covariates without elevation), trained on all Serengeti points. It's defined everywhere.
- **camera:** a Gaussian-weighted average (`CAMERA_SIGMA_M`, 2.5 km) of the buffalo photo rates of cameras within `CAMERA_REACH_M` (5 km). Further away it's left **missing**, not guessed.
- **sightings:** the effort-adjusted share from 11, smoothed instead of cut into 5 km squares: Σ weight × buffalo ÷ Σ weight × all mammal records, with Gaussian weights (`SIGHTINGS_SIGMA_M`, 5 km). It's left missing where the weighted record count is under `MIN_RECORDS` (20), too little to say.

In [2]:
CAMERA_SIGMA_M, CAMERA_REACH_M = 2_500, 5_000
SIGHTINGS_SIGMA_M, MIN_RECORDS = 5_000, 20
MODEL_FEATURES = BAND_COLS + [c for c in COVARIATE_COLS if c != "elevation_m"]
SOURCES = ["camera", "model", "sightings"]


def kernel_sums(target_xy, source_xy, source_values, sigma, reach):
    """Gaussian-weighted sums of source_values (n_sources x k) around each target, and the summed weights."""
    tree = cKDTree(source_xy)
    sums, total = np.full((len(target_xy), source_values.shape[1]), np.nan), np.zeros(len(target_xy))
    for i, idx in enumerate(tree.query_ball_point(target_xy, r=reach)):
        if idx:
            w = np.exp(-((source_xy[idx] - target_xy[i]) ** 2).sum(axis=1) / (2 * sigma ** 2))
            sums[i], total[i] = w @ source_values[idx], w.sum()
    return sums, total


def camera_score(target_xy, cameras):
    sums, total = kernel_sums(target_xy, cameras[["x", "y"]].values, cameras[["rate"]].values, CAMERA_SIGMA_M, CAMERA_REACH_M)
    return np.where(total > 0, sums[:, 0] / np.where(total > 0, total, 1), np.nan)


def sightings_score(target_xy):
    sums, total = kernel_sums(target_xy, records[["x", "y"]].values, records[["buffalo"]].values, SIGHTINGS_SIGMA_M, 3 * SIGHTINGS_SIGMA_M)
    return np.where(total >= MIN_RECORDS, sums[:, 0] / np.where(total > 0, total, 1), np.nan)


def percentile_of(values, reference):
    """Percentile (0-100) of each value within the non-missing reference values; missing stays missing."""
    ref = np.sort(reference[~np.isnan(reference)])
    return np.where(np.isnan(values), np.nan, np.searchsorted(ref, values, side="right") / len(ref) * 100)


model = bv.make_model(serengeti["presence"], bv.XGB_SETTINGS["regularized"]).fit(serengeti[MODEL_FEATURES], serengeti["presence"])
grid_xy = grid[["x", "y"]].values
raw = pd.DataFrame({
    "camera": camera_score(grid_xy, cam),
    "model": model.predict_proba(grid[MODEL_FEATURES])[:, 1],
    "sightings": sightings_score(grid_xy),
})
pct = pd.DataFrame({s: percentile_of(raw[s].values, raw[s].values) for s in SOURCES})

print("grid points where each score exists:")
print(pct.notna().sum().astype(str) + " of " + str(len(grid)))

grid points where each score exists:
camera        273 of 2693
model        2693 of 2693
sightings    1643 of 2693
dtype: str


## 2. Does the rule work? Camera regions held out

The cameras are split into the same 4 k-means regions as in 09. For each region:
- its cameras are **hidden**
- the camera score at their positions is rebuilt from the **other three regions' cameras only** (missing if none is within 5 km)
- the model and sightings scores are computed at the hidden cameras as usual

Each variant of the rule then ranks the hidden cameras, compared with their real photo rate (Spearman rho).

**Reading it:** the full rule is worth using only if it ranks hidden cameras better than the model alone. `cameras scored` shows how many hidden cameras each variant could score: the camera score only exists near other regions' cameras.

Percentiles here are computed against the grid's model and sightings scores and the training cameras' rates, the same scales the map uses.

In [3]:
WEIGHTS = {"camera": 0.5, "model": 0.3, "sightings": 0.2}


def combine(pcts, weights):
    """Weighted average over the scores available at each place, plus how many there were."""
    w = np.array([weights.get(s, 0) for s in SOURCES])
    available = pcts[SOURCES].notna().values & (w > 0)
    total = (available * w).sum(axis=1)
    score = (np.nan_to_num(pcts[SOURCES].values) * w * available).sum(axis=1) / np.where(total > 0, total, np.nan)
    return score, available.sum(axis=1)


cam["camera region"] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(cam[["x", "y"]])
held_out = []
for region in range(4):
    train, test = cam[cam["camera region"] != region], cam[cam["camera region"] == region]
    test_xy = test[["x", "y"]].values
    held_out.append(pd.DataFrame({
        "rate": test["rate"].values,
        "camera": percentile_of(camera_score(test_xy, train), train["rate"].values),
        "model": percentile_of(model.predict_proba(test[MODEL_FEATURES])[:, 1], raw["model"].values),
        "sightings": percentile_of(sightings_score(test_xy), raw["sightings"].values),
    }))
held_out = pd.concat(held_out, ignore_index=True)

VARIANTS = {
    "model only": {"model": 1},
    "sightings only": {"sightings": 1},
    "camera only (other regions)": {"camera": 1},
    "model + sightings": {"model": 0.3, "sightings": 0.2},
    "full rule (camera + model + sightings)": WEIGHTS,
}
rows = []
for name, weights in VARIANTS.items():
    score, _ = combine(held_out, weights)
    ok = ~np.isnan(score)
    rows.append({"variant": name, "cameras scored": int(ok.sum()), "Spearman rho": spearmanr(score[ok], held_out["rate"][ok])[0]})
pd.DataFrame(rows).set_index("variant").round(3)

,cameras scored,Spearman rho
variant,,
model only,222,0.564
sightings only,222,0.197
camera only (other regions),103,0.200
model + sightings,222,0.424
full rule (camera + model + sightings),222,0.443


## 3. How much do the weights matter?

0.5 / 0.3 / 0.2 is a judgement call. Here the combined map is recomputed with other reasonable weights and compared with the default:
- **rank agreement** over all grid points
- **overlap of the top 10 %**, the areas the map would actually highlight

If both stay high, the exact weights barely matter.

In [4]:
default_score, n_sources = combine(pct, WEIGHTS)
grid["evidence"], grid["sources"] = default_score, n_sources

WEIGHT_OPTIONS = {
    "equal (1/3 each)": {"camera": 1 / 3, "model": 1 / 3, "sightings": 1 / 3},
    "no sightings (0.6 / 0.4 / 0)": {"camera": 0.6, "model": 0.4, "sightings": 0},
    "model-led (0.3 / 0.5 / 0.2)": {"camera": 0.3, "model": 0.5, "sightings": 0.2},
}
top_default = default_score >= np.quantile(default_score, 0.9)
rows = []
for name, weights in WEIGHT_OPTIONS.items():
    score, _ = combine(pct, weights)
    top = score >= np.quantile(score, 0.9)
    rows.append({"weights": name, "rank agreement": spearmanr(score, default_score)[0], "top-10 % overlap": (top & top_default).sum() / top_default.sum()})
print("sources behind each grid point:", pd.Series(n_sources).value_counts().sort_index().to_dict())
pd.DataFrame(rows).set_index("weights").round(2)

sources behind each grid point: {1: 1049, 2: 1372, 3: 272}


,rank agreement,top-10 % overlap
weights,,
equal (1/3 each),0.99,0.91
no sightings (0.6 / 0.4 / 0),0.93,0.63
model-led (0.3 / 0.5 / 0.2),0.99,0.90


## 4. Map: the best places, as a few ranked areas

So that areas can't merge into one large blob, they're picked as **separate peaks**:
1. Take the grid point with the highest combined score.
2. Its area is every top-20 % point (`HOT_PERCENT`) within `PEAK_RADIUS_M` (6 km) of it.
3. Remove those points and repeat, `N_AREAS` (8) times.

Each area is drawn from its 2.5 km grid squares and numbered by rank, 1 darkest. Hover an area for its size, score and how many sources back it. An area backed by cameras rests on stronger evidence than one backed by the model and sightings only.

In [5]:
import folium
import geopandas as gpd
from shapely.geometry import box
from shapely.ops import unary_union

HOT_PERCENT = 20
PEAK_RADIUS_M = 6_000
N_AREAS = 8

remaining = grid[grid["evidence"] >= np.quantile(grid["evidence"], 1 - HOT_PERCENT / 100)].copy()
picked = []
while len(picked) < N_AREAS and len(remaining):
    peak = remaining.loc[remaining["evidence"].idxmax()]
    members = remaining[np.hypot(remaining["x"] - peak["x"], remaining["y"] - peak["y"]) <= PEAK_RADIUS_M]
    picked.append((peak, members))
    remaining = remaining.drop(members.index)

half = GRID_SPACING_M / 2
source_names = {3: "camera + model + sightings", 2: "two sources", 1: "model only"}
area_table = gpd.GeoDataFrame([
    {"rank": rank, "area_km2": len(members) * (GRID_SPACING_M / 1000) ** 2,
     "score": members["evidence"].mean(), "sources": int(peak["sources"]),
     "backed by": "cameras + " + ("sightings + model" if peak["sources"] == 3 else "model") if not np.isnan(pct.loc[peak.name, "camera"])
                  else ("model + sightings" if peak["sources"] == 2 else "model only"),
     "geometry": unary_union([box(x - half, y - half, x + half, y + half) for x, y in members[["x", "y"]].values]),
     "peak_lon": peak["decimalLongitude"], "peak_lat": peak["decimalLatitude"]}
    for rank, (peak, members) in enumerate(picked, start=1)
], geometry="geometry", crs=bd.UTM).to_crs("EPSG:4326")

print(area_table[["rank", "area_km2", "score", "backed by", "peak_lon", "peak_lat"]].round(3).to_string(index=False))

light, dark = np.array([0x86, 0xb6, 0xef]), np.array([0x0d, 0x36, 0x6b])
colour = lambda rank: "#%02x%02x%02x" % tuple((dark + (light - dark) * (rank - 1) / max(len(area_table) - 1, 1)).astype(int))

evidence_map = folium.Map(location=[grid["decimalLatitude"].mean(), grid["decimalLongitude"].mean()], zoom_start=8,
                          tiles="OpenStreetMap")
for _, a in area_table.iterrows():
    c = colour(a["rank"])
    folium.GeoJson(a.geometry.__geo_interface__,
                   style_function=lambda _, c=c: {"fillColor": c, "color": c, "weight": 1.5, "fillOpacity": 0.55},
                   tooltip=f"area {a['rank']}: {a['area_km2']:.0f} km², score {a['score']:.0f}, backed by {a['backed by']}").add_to(evidence_map)
    folium.Marker([a["peak_lat"], a["peak_lon"]], icon=folium.DivIcon(
        html=f'<div style="font:bold 13px sans-serif;color:#fff;background:{c};border-radius:50%;width:22px;height:22px;'
             f'line-height:22px;text-align:center;border:1.5px solid #fff">{a["rank"]}</div>',
        icon_size=(22, 22), icon_anchor=(11, 11))).add_to(evidence_map)
evidence_map

 rank  area_km2  score         backed by  peak_lon  peak_lat
    1     25.00 88.721        model only    34.383    -2.793
    2    118.75 88.366        model only    35.147    -2.294
    3     56.25 81.395        model only    34.517    -2.137
    4     31.25 81.608        model only    34.540    -2.566
    5     68.75 85.940        model only    35.102    -2.249
    6     18.75 86.719        model only    34.428    -2.612
    7     37.50 80.418        model only    35.124    -2.159
    8    131.25 87.053 model + sightings    34.944    -2.068


**Reading it, honestly:**
- **The ranking is relative:** area 1 is where the evidence is strongest, compared with the rest of the visited Serengeti. It's not a probability of seeing buffalo.
- **"backed by"** shows why an area ranks high. Camera-backed areas rest on effort-corrected data, but from 2010–2013. Areas backed only by the model and sightings depend on tourist data and the landscape model.
- **Not covered:** places outside the visited Serengeti, the crater highlands (see 11 for those), and anything finer than the 2.5 km grid.